# Análisis de calidad de los datos oficiales

## tl;dr

Las fuentes contienen volumen suficiente para una demostración completa y
errores intencionales controlados. No reducimos Bronze: Silver explica cada
fila y Gold conserva los conteos y métricas de negocio de referencia.

In [1]:
from pathlib import Path
import json
import re
import pandas as pd
from IPython.display import display

RAIZ = Path.cwd().resolve()
if not (RAIZ / "datos" / "entrada").exists():
    RAIZ = RAIZ.parent
ENTRADA = RAIZ / "datos" / "entrada"
nombres = ["sucursales", "productos", "clientes", "cambios_cliente",
           "cambios_producto", "ventas", "devoluciones"]
tablas = {
    nombre: pd.read_csv(ENTRADA / f"{nombre}.csv", dtype=str, keep_default_na=False)
    for nombre in nombres
}

## Perfil y grano

In [2]:
perfil = pd.DataFrame([
    {
        "fuente": nombre,
        "filas": len(datos),
        "columnas": len(datos.columns),
        "duplicados_exactos": int(datos.duplicated().sum()),
    }
    for nombre, datos in tablas.items()
])
display(perfil)

,fuente,filas,columnas,duplicados_exactos
0,sucursales,22,8,0
1,productos,1000,7,0
2,clientes,10100,5,0
3,cambios_cliente,2000,4,0
4,cambios_producto,50,6,0
5,ventas,125854,13,626
6,devoluciones,4618,8,1


## Identidad, referencias y dominios

In [3]:
clientes = tablas["clientes"].assign(
    id_norm=lambda x: x.id_cliente.str.strip().str.upper(),
    correo_norm=lambda x: x.correo.str.strip().str.lower(),
)
patron_correo = re.compile(r"^[^\s@]+@[^\s@]+\.[^\s@]+$")
ventas = tablas["ventas"]
hallazgos = pd.DataFrame([
    ("clientes duplicados normalizados", int(clientes.duplicated("id_norm", keep=False).sum())),
    ("correos duplicados normalizados", int(clientes.duplicated("correo_norm", keep=False).sum())),
    ("correos con formato inválido", int((~clientes.correo_norm.map(lambda v: bool(patron_correo.fullmatch(v)))).sum())),
    ("ventas con cliente desconocido", int((~ventas.id_cliente.str.strip().str.upper().isin(set(clientes.id_norm))).sum())),
    ("ventas con producto desconocido", int((~ventas.codigo_producto.str.strip().str.upper().isin(set(tablas["productos"].codigo_producto.str.strip().str.upper()))).sum())),
    ("ventas con sucursal desconocida", int((~ventas.codigo_sucursal.str.strip().str.upper().isin(set(tablas["sucursales"].codigo_sucursal.str.strip().str.upper()))).sum())),
], columns=["control", "filas"])
display(hallazgos)

,control,filas
0,clientes duplicados normalizados,200
1,correos duplicados normalizados,200
2,correos con formato inválido,0
3,ventas con cliente desconocido,1
4,ventas con producto desconocido,1
5,ventas con sucursal desconocida,1


## Fechas y medidas de ventas

In [4]:
fecha = pd.to_datetime(ventas.fecha_venta, errors="coerce")
medidas = {
    "fecha_venta_invalida": int(fecha.isna().sum()),
    "cantidad_no_positiva": int((pd.to_numeric(ventas.cantidad, errors="coerce") <= 0).sum()),
    "precio_no_numerico": int(pd.to_numeric(ventas.precio_unitario, errors="coerce").isna().sum()),
    "descuento_fuera_rango": int((~pd.to_numeric(ventas.porcentaje_descuento, errors="coerce").between(0, 100)).sum()),
}
display(pd.Series(medidas, name="filas").to_frame())
print("Período válido:", fecha.min().date(), "a", fecha.max().date())

,filas
fecha_venta_invalida,1
cantidad_no_positiva,1
precio_no_numerico,1
descuento_fuera_rango,1


Período válido: 2024-01-01 a 2025-12-31


## Auditoría Silver ejecutada

In [5]:
ruta_auditoria = RAIZ / "resultados" / "silver" / "auditoria_silver.json"
auditoria = json.loads(ruta_auditoria.read_text(encoding="utf-8"))
display(pd.DataFrame(auditoria["conciliacion"]))
rechazos = pd.DataFrame(auditoria["rechazos"])
display(rechazos[["fuente", "motivo"]].value_counts().reset_index(name="filas"))
assert all(fila["diferencia"] == 0 for fila in auditoria["conciliacion"])

,fuente,filas_leidas,filas_aceptadas,filas_rechazadas,filas_duplicadas,diferencia
0,clientes,10100,10000,0,100,0
1,cambios_cliente,2000,2000,0,0,0
2,productos,1000,1000,0,0,0
3,cambios_producto,50,50,0,0,0
4,sucursales,22,22,0,0,0
5,ventas,125854,125217,11,626,0
6,devoluciones,4618,4612,5,1,0


,fuente,motivo,filas
0,ventas,conflicto_clave_negocio,2
1,ventas,campo_obligatorio_vacio,1
2,ventas,producto_desconocido,1
3,ventas,medida_no_positiva,1
4,ventas,fecha_venta_invalida,1
5,ventas,descuento_invalido,1
6,ventas,importe_incoherente,1
7,ventas,cliente_desconocido,1
8,ventas,estado_no_admitido,1
9,ventas,valor_numerico_invalido,1


## Muestra legible de correos

Esta muestra separada usa nombres ficticios plausibles. Sirve para enseñar
formato, vacío y duplicidad sin modificar los CSV oficiales ni sus métricas.

In [6]:
muestra = pd.read_csv(
    RAIZ / "datos" / "muestra_calidad" / "clientes_muestra.csv",
    dtype=str,
    keep_default_na=False,
)
muestra["correo_normalizado"] = muestra.correo.str.strip().str.lower()
muestra["formato_valido"] = muestra.correo_normalizado.map(
    lambda valor: bool(patron_correo.fullmatch(valor))
)
muestra["correo_duplicado"] = muestra.correo_normalizado.ne("") & muestra.correo_normalizado.duplicated(keep=False)
display(muestra[["id_cliente", "nombre_cliente", "correo", "formato_valido", "correo_duplicado"]])
display(pd.Series({
    "formato_valido": int(muestra.formato_valido.sum()),
    "formato_invalido_o_vacio": int((~muestra.formato_valido).sum()),
    "filas_con_correo_duplicado": int(muestra.correo_duplicado.sum()),
}, name="filas").to_frame())

,id_cliente,nombre_cliente,correo,formato_valido,correo_duplicado
0,M001,Camila Soto,camila.soto@example.invalid,True,True
1,M002,Matías Rojas,matias.rojas@example.invalid,True,False
2,M003,Valentina Muñoz,valentina.munoz@example.invalid,True,False
3,M004,Sebastián Díaz,sebastian.diaz.example.invalid,False,False
4,M005,Francisca Silva,francisca.silva@example.invalid,True,False
5,M006,Tomás Herrera,tomas@@example.invalid,False,False
6,M007,Josefa Morales,josefa.morales@example.invalid,True,False
7,M008,Nicolás Castro,nicolas.castro@example.invalid,True,False
8,M009,Martina Torres,martina.torres@example.invalid,True,False
9,M010,Benjamín Reyes,,False,False


,filas
formato_valido,8
formato_invalido_o_vacio,4
filas_con_correo_duplicado,2


## Takeaways

- El volumen es manejable y cubre suficientes casos de calidad; no se reduce.
- Los correos son sintéticos y válidos, con dominio reservado.
- Los errores intencionales permanecen en Bronze y quedan explicados por Silver.
- La separación de capas permite enseñar limpieza antes de construir dimensiones.